In [1]:
# QKRR baseline for `crossgen_biosignatures_20260311`
#
# This notebook builds a subset-aware quantum kernel ridge regression (QKRR)
# benchmark for the cross-generator biosignatures dataset.
#
# Benchmark contract:
# - Inputs come from `spectra.h5` using `transit_depth_noisy` plus `sigma_ppm`.
# - Regression targets come from `labels.parquet` using the five `log10_vmr_*` columns.
# - The split is fixed: `tau/train`, `tau/val`, `poseidon/test`.
# - The main reporting metric is per-target RMSE on validation and test.
#
# Because exact quantum kernels scale poorly with sample count, this notebook
# keeps dimensionality reduction and reproducible subset sampling as first-class
# configuration options rather than afterthoughts.

In [2]:
# Optional bootstrap for a fresh notebook kernel.
# Uncomment the next line if the active kernel is missing dependencies.
# %pip install numpy pandas pyarrow h5py scikit-learn matplotlib tqdm qiskit qiskit-machine-learning qiskit-algorithms

In [3]:
from __future__ import annotations

import importlib.util
import random
import sys
import time
import warnings
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)


def _module_exists(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None

/Users/jkw/Documents/uni/axion/hack4sages/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "crossgen_biosignatures_20260311"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Expected dataset directory at {DATA_DIR}. Run the notebook from the repository root."
    )

TARGET_COLUMNS = [
    "log10_vmr_h2o",
    "log10_vmr_co2",
    "log10_vmr_co",
    "log10_vmr_ch4",
    "log10_vmr_nh3",
]

PRESENT_COLUMNS = [
    "present_h2o",
    "present_co2",
    "present_co",
    "present_ch4",
    "present_nh3",
]

CONFIG = {
    "train_subset": 1024,
    "val_subset": 512,
    "test_subset": None,
    "quantum_input_dim": 12,
    "scale_targets": True,
    "classical_alpha": 1.0,
    "classical_gamma": None,
    "quantum_alpha": 1.0,
    "feature_map_reps": 2,
    "feature_map_entanglement": "linear",
    "kernel_batch_size": 16,
    "diagnostic_targets": ["log10_vmr_h2o", "log10_vmr_co2"],
}

print(f"Using dataset: {DATA_DIR}")
print(f"Quantum input dimension: {CONFIG['quantum_input_dim']}")
print(f"Subset config: train={CONFIG['train_subset']}, val={CONFIG['val_subset']}, test={CONFIG['test_subset']}")
print(f"Quantum kernel batch size: {CONFIG['kernel_batch_size']}")

Using dataset: /Users/jkw/Documents/uni/axion/hack4sages/crossgen_biosignatures_20260311
Quantum input dimension: 12
Subset config: train=1024, val=512, test=None
Quantum kernel batch size: 16


In [5]:
def _decode_utf8(values: np.ndarray) -> np.ndarray:
    decoded = []
    for value in values:
        if isinstance(value, (bytes, np.bytes_)):
            decoded.append(value.decode("utf-8"))
        else:
            decoded.append(str(value))
    return np.asarray(decoded, dtype=object)


def _sample_indices(indices: np.ndarray, limit: int | None, rng: np.random.Generator) -> np.ndarray:
    if limit is None or limit >= len(indices):
        return np.sort(indices)
    return np.sort(rng.choice(indices, size=limit, replace=False))


def _ensure_parquet_engine() -> None:
    pyarrow_available = _module_exists("pyarrow")
    fastparquet_available = _module_exists("fastparquet")

    if not (pyarrow_available or fastparquet_available):
        raise ImportError(
            "No parquet engine is installed in the active notebook kernel. "
            "Run `%pip install pyarrow` in this notebook, restart the kernel, and rerun from the top. "
            f"Active kernel executable: {sys.executable}"
        )


def load_crossgen_regression_problem(data_dir: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    spectra_path = data_dir / "spectra.h5"
    labels_path = data_dir / "labels.parquet"
    label_columns = ["sample_id", "generator", "split", *TARGET_COLUMNS, *PRESENT_COLUMNS]

    with h5py.File(spectra_path, "r") as handle:
        spectra = np.asarray(handle["transit_depth_noisy"], dtype=np.float32)
        sigma_ppm = np.asarray(handle["sigma_ppm"], dtype=np.float32).reshape(-1, 1)
        sample_id = _decode_utf8(handle["sample_id"][...])
        generator = _decode_utf8(handle["generator"][...])
        split = _decode_utf8(handle["split"][...])

    _ensure_parquet_engine()
    labels = pd.read_parquet(labels_path, columns=label_columns)

    label_sample_id = labels["sample_id"].astype(str).to_numpy()
    if not np.array_equal(sample_id, label_sample_id):
        raise ValueError(
            "labels.parquet and spectra.h5 are not aligned in this environment; join on sample_id before proceeding."
        )

    X = np.concatenate([spectra, sigma_ppm], axis=1).astype(np.float32)
    y = labels[TARGET_COLUMNS].to_numpy(dtype=np.float32)
    present = labels[PRESENT_COLUMNS].to_numpy(dtype=np.int8)

    metadata = pd.DataFrame(
        {
            "sample_id": sample_id,
            "generator": generator,
            "split": split,
        }
    )

    return X, y, present, metadata


def make_split_subsets(metadata: pd.DataFrame, config: dict, seed: int) -> dict[str, np.ndarray]:
    rng = np.random.default_rng(seed)
    split_arrays = {
        split_name: np.flatnonzero(metadata["split"].to_numpy() == split_name)
        for split_name in ["train", "val", "test"]
    }

    sampled = {
        "train": _sample_indices(split_arrays["train"], config["train_subset"], rng),
        "val": _sample_indices(split_arrays["val"], config["val_subset"], rng),
        "test": _sample_indices(split_arrays["test"], config["test_subset"], rng),
    }
    return sampled

In [6]:
X_all, y_all, present_all, metadata = load_crossgen_regression_problem(DATA_DIR)
subset_indices = make_split_subsets(metadata, CONFIG, RANDOM_SEED)

split_summary = metadata.groupby(["generator", "split"]).size().rename("rows")
display(split_summary.to_frame())

assert set(metadata.loc[metadata["split"] == "train", "generator"]) == {"tau"}
assert set(metadata.loc[metadata["split"] == "val", "generator"]) == {"tau"}
assert set(metadata.loc[metadata["split"] == "test", "generator"]) == {"poseidon"}

X_train = X_all[subset_indices["train"]]
X_val = X_all[subset_indices["val"]]
X_test = X_all[subset_indices["test"]]

y_train = y_all[subset_indices["train"]]
y_val = y_all[subset_indices["val"]]
y_test = y_all[subset_indices["test"]]

present_train = present_all[subset_indices["train"]]
present_val = present_all[subset_indices["val"]]
present_test = present_all[subset_indices["test"]]

print("Feature matrix shape (full):", X_all.shape)
print("Feature matrix shape (sampled train/val/test):", X_train.shape, X_val.shape, X_test.shape)
print("Target matrix shape (full):", y_all.shape)
print("First five target columns:", TARGET_COLUMNS)
print("Using baseline-compatible feature_dim =", X_train.shape[1])

rows
generator split       
poseidon  test     685
tau       train  37281
          val     4142

Feature matrix shape (full): (42108, 219)
Feature matrix shape (sampled train/val/test): (1024, 219) (512, 219) (685, 219)
Target matrix shape (full): (42108, 5)
First five target columns: ['log10_vmr_h2o', 'log10_vmr_co2', 'log10_vmr_co', 'log10_vmr_ch4', 'log10_vmr_nh3']
Using baseline-compatible feature_dim = 219


In [7]:
def fit_preprocessors(
    X_train: np.ndarray,
    X_eval: dict[str, np.ndarray],
    y_train: np.ndarray,
    n_components: int,
    scale_targets: bool,
    seed: int,
):
    x_scaler = StandardScaler()
    X_train_scaled = x_scaler.fit_transform(X_train)

    pca = PCA(n_components=n_components, random_state=seed)
    X_train_reduced = pca.fit_transform(X_train_scaled)

    transformed = {"train": X_train_reduced}
    for split_name, X_split in X_eval.items():
        transformed[split_name] = pca.transform(x_scaler.transform(X_split))

    y_scaler = None
    y_train_processed = y_train.copy()
    if scale_targets:
        y_scaler = StandardScaler()
        y_train_processed = y_scaler.fit_transform(y_train)

    prep_artifacts = {
        "x_scaler": x_scaler,
        "pca": pca,
        "explained_variance_ratio": float(np.sum(pca.explained_variance_ratio_)),
        "y_scaler": y_scaler,
    }
    return transformed, y_train_processed, prep_artifacts


def invert_target_transform(values: np.ndarray, y_scaler: StandardScaler | None) -> np.ndarray:
    if y_scaler is None:
        return values
    return y_scaler.inverse_transform(values)


X_reduced, y_train_processed, prep = fit_preprocessors(
    X_train=X_train,
    X_eval={"val": X_val, "test": X_test},
    y_train=y_train,
    n_components=CONFIG["quantum_input_dim"],
    scale_targets=CONFIG["scale_targets"],
    seed=RANDOM_SEED,
)

print("Reduced train/val/test shapes:", X_reduced["train"].shape, X_reduced["val"].shape, X_reduced["test"].shape)
print("PCA explained variance ratio:", f"{prep['explained_variance_ratio']:.4f}")

Reduced train/val/test shapes: (1024, 12) (512, 12) (685, 12)
PCA explained variance ratio: 1.0000


In [8]:
def per_target_rmse(y_true: np.ndarray, y_pred: np.ndarray, target_columns: list[str]) -> dict[str, float]:
    return {
        target: float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
        for i, target in enumerate(target_columns)
    }


def summarize_metrics(label: str, y_true_by_split: dict[str, np.ndarray], y_pred_by_split: dict[str, np.ndarray]) -> pd.DataFrame:
    rows = []
    for split_name, y_true in y_true_by_split.items():
        y_pred = y_pred_by_split[split_name]
        rmse = per_target_rmse(y_true, y_pred, TARGET_COLUMNS)
        rmse["split"] = split_name
        rmse["mean_rmse"] = float(np.mean(list(per_target_rmse(y_true, y_pred, TARGET_COLUMNS).values())))
        rmse["model"] = label
        rows.append(rmse)
    return pd.DataFrame(rows)


def fit_targetwise_kernel_ridge(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_eval: dict[str, np.ndarray],
    alpha: float,
    gamma: float | None,
) -> dict[str, np.ndarray]:
    if gamma is None:
        gamma = 1.0 / X_train.shape[1]

    predictions = {
        split_name: np.zeros((X_split.shape[0], y_train.shape[1]), dtype=np.float64)
        for split_name, X_split in X_eval.items()
    }

    for target_index, target_name in enumerate(TARGET_COLUMNS):
        model = KernelRidge(alpha=alpha, kernel="rbf", gamma=gamma)
        model.fit(X_train, y_train[:, target_index])
        for split_name, X_split in X_eval.items():
            predictions[split_name][:, target_index] = model.predict(X_split)
        print(f"Fitted classical KRR target: {target_name}")

    return predictions


classical_start = time.perf_counter()
classical_pred_scaled = fit_targetwise_kernel_ridge(
    X_train=X_reduced["train"],
    y_train=y_train_processed,
    X_eval={"val": X_reduced["val"], "test": X_reduced["test"]},
    alpha=CONFIG["classical_alpha"],
    gamma=CONFIG["classical_gamma"],
)
classical_elapsed = time.perf_counter() - classical_start

classical_pred = {
    split_name: invert_target_transform(values, prep["y_scaler"])
    for split_name, values in classical_pred_scaled.items()
}

classical_metrics = summarize_metrics(
    label="classical_krr",
    y_true_by_split={"val": y_val, "test": y_test},
    y_pred_by_split=classical_pred,
)

print(f"Classical baseline runtime: {classical_elapsed:.2f}s")
display(classical_metrics)

Fitted classical KRR target: log10_vmr_h2o
Fitted classical KRR target: log10_vmr_co2
Fitted classical KRR target: log10_vmr_co
Fitted classical KRR target: log10_vmr_ch4
Fitted classical KRR target: log10_vmr_nh3
Classical baseline runtime: 0.09s


,log10_vmr_h2o,log10_vmr_co2,log10_vmr_co,log10_vmr_ch4,log10_vmr_nh3,split,mean_rmse,model
0,2.880152,2.963459,2.956993,2.929231,2.882993,val,2.922565,classical_krr
1,2.906204,2.982215,2.860013,2.973424,2.862988,test,2.916969,classical_krr


In [9]:
def build_fidelity_quantum_kernel(feature_dimension: int, reps: int, entanglement: str):
    try:
        from qiskit.circuit.library import zz_feature_map
        from qiskit_machine_learning.kernels import FidelityQuantumKernel, FidelityStatevectorKernel
    except ImportError as exc:
        raise ImportError(
            "QKRR requires qiskit, qiskit-machine-learning, and qiskit-algorithms in the active notebook kernel."
        ) from exc

    feature_map = zz_feature_map(
        feature_dimension=feature_dimension,
        reps=reps,
        entanglement=entanglement,
    )

    # Prefer the exact statevector kernel when available: it is much faster for
    # local simulation than the generic fidelity primitive path.
    try:
        return FidelityStatevectorKernel(feature_map=feature_map)
    except Exception:
        pass

    try:
        return FidelityQuantumKernel(feature_map=feature_map)
    except Exception:
        pass

    try:
        from qiskit_algorithms.state_fidelities import ComputeUncompute
    except ImportError:
        from qiskit_machine_learning.state_fidelities import ComputeUncompute

    try:
        from qiskit.primitives import StatevectorSampler

        sampler = StatevectorSampler()
    except ImportError:
        try:
            from qiskit.primitives import Sampler

            sampler = Sampler()
        except ImportError as exc:
            raise ImportError(
                "Could not import a compatible Qiskit sampler primitive for FidelityQuantumKernel."
            ) from exc

    fidelity = ComputeUncompute(sampler=sampler)
    return FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)


def evaluate_kernel_matrix(
    kernel,
    x_left: np.ndarray,
    x_right: np.ndarray | None = None,
    batch_size: int = 16,
    desc: str = "kernel",
) -> tuple[np.ndarray, float]:
    if x_right is None:
        x_right = x_left

    if batch_size <= 0:
        raise ValueError("batch_size must be a positive integer")

    blocks = []
    total_batches = (len(x_left) + batch_size - 1) // batch_size
    start = time.perf_counter()

    for batch_start in tqdm(
        range(0, len(x_left), batch_size),
        total=total_batches,
        desc=desc,
        unit="batch",
    ):
        batch_stop = min(batch_start + batch_size, len(x_left))
        block = kernel.evaluate(x_vec=x_left[batch_start:batch_stop], y_vec=x_right)
        blocks.append(np.asarray(block, dtype=np.float64))

    matrix = np.vstack(blocks)
    elapsed = time.perf_counter() - start
    return matrix, elapsed


def fit_precomputed_qkrr(
    K_train: np.ndarray,
    y_train: np.ndarray,
    kernel_eval_mats: dict[str, np.ndarray],
    alpha: float,
) -> dict[str, np.ndarray]:
    predictions = {
        split_name: np.zeros((matrix.shape[0], y_train.shape[1]), dtype=np.float64)
        for split_name, matrix in kernel_eval_mats.items()
    }

    for target_index, target_name in tqdm(
        list(enumerate(TARGET_COLUMNS)),
        total=len(TARGET_COLUMNS),
        desc="QKRR targets",
        unit="target",
    ):
        model = KernelRidge(alpha=alpha, kernel="precomputed")
        model.fit(K_train, y_train[:, target_index])
        for split_name, matrix in kernel_eval_mats.items():
            predictions[split_name][:, target_index] = model.predict(matrix)
        print(f"Fitted QKRR target: {target_name}")

    return predictions

In [ ]:
quantum_kernel = build_fidelity_quantum_kernel(
    feature_dimension=X_reduced["train"].shape[1],
    reps=CONFIG["feature_map_reps"],
    entanglement=CONFIG["feature_map_entanglement"],
)

print("Building train kernel matrix...")
K_train, train_kernel_time = evaluate_kernel_matrix(
    quantum_kernel,
    X_reduced["train"],
    batch_size=CONFIG["kernel_batch_size"],
    desc="Train kernel",
)

print("Building validation kernel matrix...")
K_val, val_kernel_time = evaluate_kernel_matrix(
    quantum_kernel,
    X_reduced["val"],
    X_reduced["train"],
    batch_size=CONFIG["kernel_batch_size"],
    desc="Validation kernel",
)

print("Building test kernel matrix...")
K_test, test_kernel_time = evaluate_kernel_matrix(
    quantum_kernel,
    X_reduced["test"],
    X_reduced["train"],
    batch_size=CONFIG["kernel_batch_size"],
    desc="Test kernel",
)

K_train = 0.5 * (K_train + K_train.T)
np.fill_diagonal(K_train, 1.0)

print("Kernel shapes:", {"train": K_train.shape, "val": K_val.shape, "test": K_test.shape})
print(
    "Kernel timings (s):",
    {
        "train": round(train_kernel_time, 2),
        "val": round(val_kernel_time, 2),
        "test": round(test_kernel_time, 2),
    },
)

print("Solving target-wise QKRR models...")
quantum_start = time.perf_counter()
quantum_pred_scaled = fit_precomputed_qkrr(
    K_train=K_train,
    y_train=y_train_processed,
    kernel_eval_mats={"val": K_val, "test": K_test},
    alpha=CONFIG["quantum_alpha"],
)
quantum_elapsed = time.perf_counter() - quantum_start

quantum_pred = {
    split_name: invert_target_transform(values, prep["y_scaler"])
    for split_name, values in quantum_pred_scaled.items()
}

quantum_metrics = summarize_metrics(
    label="qkrr",
    y_true_by_split={"val": y_val, "test": y_test},
    y_pred_by_split=quantum_pred,
)

print(f"QKRR regression runtime: {quantum_elapsed:.2f}s")
display(quantum_metrics)

Building train kernel matrix...


Train kernel: 100%|██████████| 64/64 [04:40<00:00,  4.38s/batch]


Building validation kernel matrix...


Validation kernel: 100%|█████| 32/32 [02:21<00:00,  4.43s/batch]


Building test kernel matrix...


Test kernel:  23%|██▌        | 10/43 [00:44<02:26,  4.43s/batch]

In [ ]:
all_metrics = pd.concat([classical_metrics, quantum_metrics], ignore_index=True)
display(all_metrics)

experiment_summary = pd.DataFrame(
    [
        {
            "train_rows": len(subset_indices["train"]),
            "val_rows": len(subset_indices["val"]),
            "test_rows": len(subset_indices["test"]),
            "feature_dim": X_train.shape[1],
            "quantum_input_dim": X_reduced["train"].shape[1],
            "pca_explained_variance_ratio": prep["explained_variance_ratio"],
            "feature_map_reps": CONFIG["feature_map_reps"],
            "feature_map_entanglement": CONFIG["feature_map_entanglement"],
            "classical_runtime_s": classical_elapsed,
            "qkrr_runtime_s": quantum_elapsed,
        }
    ]
)
display(experiment_summary)

for target_name in CONFIG["diagnostic_targets"]:
    target_index = TARGET_COLUMNS.index(target_name)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=False, sharey=False)

    axes[0].scatter(y_val[:, target_index], quantum_pred["val"][:, target_index], alpha=0.6, s=18)
    axes[0].plot(
        [y_val[:, target_index].min(), y_val[:, target_index].max()],
        [y_val[:, target_index].min(), y_val[:, target_index].max()],
        linestyle="--",
        color="black",
        linewidth=1,
    )
    axes[0].set_title(f"QKRR val: {target_name}")
    axes[0].set_xlabel("true")
    axes[0].set_ylabel("pred")

    axes[1].scatter(y_test[:, target_index], quantum_pred["test"][:, target_index], alpha=0.6, s=18)
    axes[1].plot(
        [y_test[:, target_index].min(), y_test[:, target_index].max()],
        [y_test[:, target_index].min(), y_test[:, target_index].max()],
        linestyle="--",
        color="black",
        linewidth=1,
    )
    axes[1].set_title(f"QKRR test: {target_name}")
    axes[1].set_xlabel("true")
    axes[1].set_ylabel("pred")

    plt.tight_layout()
    plt.show()